## **0. Settings**

In [3]:
import os, sys
sys.path.append(os.path.abspath('..'))

## **1. Import Libraries**

In [14]:
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from src.training import data_loader, model
from src.evaluation import inference

## **2. Load Model**

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
# Load best checkpoint
model =  model.AIContentModel(model_name='Qualcomm-AI-Research/BamiBERT', num_classes=2)
checkpoint = torch.load('../models/best_model.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AIContentModel(
  (bert): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(20481, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(2050, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerN

## **3. Load Data**

In [7]:
test_df = pd.read_csv('../data/subsets/test_df.csv')
test_dataset = torch.load('../data/encoded/test_dataset.pt', map_location=device, weights_only=False)
test_loader = data_loader.create_dataloader(test_dataset, batch_size=4, shuffle=False)

## **4. Run Inference**

In [9]:
all_preds, all_labels, all_probs = inference.predict(model, test_loader, device)
test_df['prediction'] = all_preds
test_df['prob_ai'] = all_probs

## **5. Overall Metrics**

In [12]:
accuracy = accuracy_score(all_preds, all_labels)
print('Accuracy score:', round(accuracy, 2))

Accuracy score: 0.93


In [15]:
report = classification_report(all_preds, all_labels)
print(report)

              precision    recall  f1-score   support

           0       0.99      0.88      0.93       429
           1       0.87      0.99      0.93       337

    accuracy                           0.93       766
   macro avg       0.93      0.94      0.93       766
weighted avg       0.94      0.93      0.93       766

